In [ ]:
from openai import OpenAI
import os
import pandas as pd
import base64
import io
from PIL import Image
import os
import json
import re

# Point to the local server
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
print(client.models.list())
model_name = 'qwen/qwen3-vl-30b'
output_dir = f"results_{model_name}"
character_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "Judgment",
        "schema": {
            "type": "object",
            "properties": {
                "Judgment": {"type": "string"},
                "Reasons": {"type": "string"}
            },
            "required": ["Judgment","Reasons"]
        },
    }
}
def escape_inner_quotes(json_str):
    reasons_match = re.search(r'"Reasons"\s*:\s*"(.*?)"', json_str, re.DOTALL)
    if reasons_match:
        original_reasons = reasons_match.group(1)
        cleaned_reasons = original_reasons.replace('"', '')
        json_str = json_str.replace(original_reasons, cleaned_reasons)
    return json_str
# Create the folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Load metadata
prompt_method = "Zero-shot1-GEO"
metadata_df = pd.read_csv("merged_metadata.csv")
image_dir = "streetview"
results = []

for _, row in metadata_df.iterrows():
    merged_index = row["merged_index"]
    study_question = row["study_question"]
    left_place = row["place_name_left"]
    right_place = row["place_name_right"]
    ground_truth = str(row["choice"]).strip().lower()
    image_path = os.path.join(image_dir, f"merged_{merged_index:04d}.jpg")

    # Encode image to base64
    image = Image.open(image_path).convert("RGB")
    img_byte_arr = io.BytesIO()
    image.save(img_byte_arr, format="JPEG")
    base64_image = base64.b64encode(img_byte_arr.getvalue()).decode("utf-8")

    # Compose LLM prompt
    prompt_text = f"""
                    You are shown a side-by-side image with two street views at two different cities: the left half at {left_place} and the right half at {right_place}.
                    Which side looks more {study_question}?

                    Answer with only one word: left or right. Then explain your reasoning.

                    Format:
                    Judgment. Reasons.
                    """

    # Call LLM 
    response = client.chat.completions.create(
        model= model_name,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text.strip()},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
                ],
            },
        ],
        response_format=character_schema,
    )
    raw = response.choices[0].message.content.strip()
    raw = (
    raw.replace("“", '"')
       .replace("”", '"')
       .replace("‘", "'")
       .replace("’", "'")
    )
    
    # Step 2: Load JSON safely
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            json_str = match.group(0)
            parsed = json.loads(json_str)
            model_judgement = parsed.get("Judgment", "").strip().lower()
            model_reason = parsed.get("Reasons", "").strip()
        except json.JSONDecodeError:
            #print("❌ JSON found but failed to parse. Falling back.")
            model_judgement = ""
            model_reason = ""
    else:
        # Fallback: Extract via regex
        #print("❌ No valid JSON object found in model output.")
        #print("🔎 Raw content was:\n", raw)
        model_judgement = ""
        model_reason = ""

        fallback_match = re.search(r'Judgment:\s*(\w+).*?Reasons:\s*(.+)', raw, re.DOTALL | re.IGNORECASE)
        if fallback_match:
            model_judgement = fallback_match.group(1).strip().lower()
            model_reason = fallback_match.group(2).strip()
    print(model_judgement,model_reason)

    results.append({
        "merged_index": merged_index,
        "left": row["left"],
        "right": row["right"],
        "study_question": study_question,
        "ground_truth": ground_truth,
        "left_vote":row["left_vote"],
        "right_vote":row["right_vote"],
        "model_judgement": model_judgement,
        "model_reason": model_reason,
        "validation": int(model_judgement == ground_truth)
    })

# Save results
df_result = pd.DataFrame(results)
df_result.to_csv(f"{output_dir}/llm_predictions_{model_name}_{prompt_method}.csv", index=False)

# Print accuracy
# Accuracy including all responses
accuracy_all = df_result["validation"].mean()

# Accuracy excluding any 'equal' in ground truth or model judgement
filtered_df = df_result[
    (df_result["ground_truth"] != "equal") & 
    (df_result["model_judgement"] != "equal")
]
accuracy_excl_equal = filtered_df["validation"].mean()

# Print both
print(f"✅ Accuracy (all): {accuracy_all:.2%}")
print(f"✅ Accuracy (excluding 'equal'): {accuracy_excl_equal:.2%}")